# ZWO vs Ximea — Camera Performance Comparison

Head-to-head comparison of the **Ximea** (BGGR, 69 nm pixel, reference) and **ZWO** (RGGB, 71.5 nm pixel, peak QE 0.91) cameras using the `STANDARD_ITER` fitting strategy across a range of photon levels and dyes.

Both cameras use real per-pixel calibration maps (gain, offset, variance, readnoise, RQE).  
The ZWO `pixel_QYs` curves are renormalised to a peak of **0.91** to reflect its lower peak quantum yield.

**Metrics reported:**
- $\sigma_{xy}$: localisation precision (nm), averaged over both axes
- $\sigma_{\mathrm{colour}}$: Euclidean RGB precision
- Per-channel colour precision ($\sigma_B$, $\sigma_G$, $\sigma_R$)
- $\chi^2$ goodness-of-fit statistics

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import types
import xarray as xr
from scipy.spatial.distance import cdist

import sys
sys.path.append("../../..")

from src import IOFunctions
IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions
from src.Multicolour_Simulation_Functions import FittingStrategy, CameraParameters, SimulationConfig
MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PlottingBase
plotter = PlottingBase.PublicationPlotter(dark_background=False)

from src import sCMOSFunctions
sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import SpectralFunctions
S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions
M_F = MaskFunctions.Mask_Functions()

print("Available fitting strategies:", [s.value for s in FittingStrategy])

INFO:Constants:Logging to file: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Code/Python/pyBayerSMLM/logs/Constants_20260409_141612.log


Available fitting strategies: ['standard', 'standard_iter', 'standard_data', 'demosaic', 'demosaic_fast', 'demosaic_ig', 'standard_ig']


## Camera calibrations and optical setup

In [2]:
# Spectral pixel efficiency curves (shared between cameras — same optical path)
R_eff, G_eff, B_eff, wavelength = S_F.getpixelefficiency()

image_size = 14

# ── Ximea ────────────────────────────────────────────────────────────────────
# Pattern: BGGR  |  B G |   pixel_size = 69 nm
#                |  G R |
ximea_folder = "../../../Camera_Calibrations/Ximea_Camera/"
ximea_gain      = IO.read_tiff(os.path.join(ximea_folder, "gain.tif"))
ximea_offset    = IO.read_tiff(os.path.join(ximea_folder, "offset.tif"))
ximea_variance  = IO.read_tiff(os.path.join(ximea_folder, "variance.tif"))
ximea_readnoise = IO.read_tiff(os.path.join(ximea_folder, "readnoise.tif"))
ximea_rqe       = IO.read_tiff(os.path.join(ximea_folder, "rqe.tif"))

ximea_mosaic = np.array([["B", "G"], ["G", "R"]])
ximea_masks  = M_F.get_masks(size_x=image_size, size_y=image_size, mosaic_unit=ximea_mosaic)
# mask order from dstack follows first-appearance order in mosaic: B, G, R
ximea_pixel_QYs = np.vstack([B_eff, G_eff, R_eff])   # peak QE ≈ 1.0 for Ximea

validated_ximea = CameraParameters.validate_and_create({
    "gain":               np.full((image_size, image_size), np.median(ximea_gain)),
    "offset":             np.full((image_size, image_size), np.median(ximea_offset)),
    "variance":           np.full((image_size, image_size), np.median(ximea_variance)),
    "readnoise":          np.full((image_size, image_size), np.median(ximea_readnoise)),
    "rqe":                np.full((image_size, image_size), np.median(ximea_rqe)),
    "masks":              ximea_masks,
    "pixel_QYs":          ximea_pixel_QYs,
    "pixel_order":        ["B", "G", "R"],
    "pixel_order_indices": {"B": 0, "G": 1, "R": 2},
})
print("Ximea camera parameters validated.")

Ximea camera parameters validated.


In [3]:
# ── ZWO ──────────────────────────────────────────────────────────────────────
# Pattern: RGGB  |  R G |   pixel_size = 71.5 nm   peak QE = 0.91
#                |  G B |
zwo_folder = "../../../Camera_Calibrations/ZWO_Camera/"
zwo_gain      = IO.read_tiff(os.path.join(zwo_folder, "gain.tif"))
zwo_offset    = IO.read_tiff(os.path.join(zwo_folder, "offset.tif"))
zwo_variance  = IO.read_tiff(os.path.join(zwo_folder, "variance.tif"))
zwo_readnoise = IO.read_tiff(os.path.join(zwo_folder, "readnoise.tif"))
zwo_rqe       = IO.read_tiff(os.path.join(zwo_folder, "rqe.tif"))

ZWO_PEAK_QE = 0.91

zwo_mosaic = np.array([["R", "G"], ["G", "B"]])
zwo_masks  = M_F.get_masks(size_x=image_size, size_y=image_size, mosaic_unit=zwo_mosaic)
# mask order from dstack follows first-appearance order in mosaic: R, G, B
# Renormalise pixel_QYs to ZWO peak QE of 0.91
zwo_pixel_QYs = np.vstack([R_eff, G_eff, B_eff]) * ZWO_PEAK_QE

validated_zwo = CameraParameters.validate_and_create({
    "gain":               np.full((image_size, image_size), np.median(zwo_gain)),
    "offset":             np.full((image_size, image_size), np.median(zwo_offset)),
    "variance":           np.full((image_size, image_size), np.median(zwo_variance)),
    "readnoise":          np.full((image_size, image_size), np.median(zwo_readnoise)),
    "rqe":                np.full((image_size, image_size), np.median(zwo_rqe)),
    "masks":              zwo_masks,
    "pixel_QYs":          zwo_pixel_QYs,
    "pixel_order":        ["R", "G", "B"],
    "pixel_order_indices": {"R": 0, "G": 1, "B": 2},
})
print(f"ZWO camera parameters validated (peak QE = {ZWO_PEAK_QE}).")

ZWO camera parameters validated (peak QE = 0.91).


In [4]:
# Optical path filters (same for both cameras — shared optical bench)
filters = [
    "semrock-di03-r405-488-561-635-t1-25x36",
    "semrock-nf03-405-488-561-635e",
]

# Gaussian pre-smoothing (same for both cameras)
smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

print("Optical setup configured.")

Optical setup configured.


## Simulation configuration

Common parameters shared across both cameras; pixel sizes differ (69 nm Ximea, 71.5 nm ZWO).

In [5]:
dyes = ["ATTO 488", "ATTO 565", "ATTO 647N"]

n_photon_space = np.unique(
    np.around(np.logspace(np.log10(500), np.log10(50000), 200) / 5) * 5
)

# Per-camera configs — pixel size is the only difference
_common = dict(
    n_bootstrap=100000,
    background_photons=5.0,
    background_colour=[1, 1, 1],
    NA=1.49,
    cpu_fraction=0.9,
    save_raw_results=True,
    subtractx0y0=True,
    saverawimages=False,
    verbose=False,
    use_stochastic_photons=True,
)

config_ximea = SimulationConfig(pixel_size=69,   **_common)   # nm
config_zwo   = SimulationConfig(pixel_size=71.5, **_common)   # nm

save_folder = r"/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20260409_ZWO_vs_Ximea"
os.makedirs(save_folder, exist_ok=True)
print(f"Results will be saved to: {save_folder}")
print(f"Photon levels: {len(n_photon_space)} points, {n_photon_space[0]:.0f}–{n_photon_space[-1]:.0f} photons")

Results will be saved to: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20260409_ZWO_vs_Ximea
Photon levels: 200 points, 500–50000 photons


## Run simulations — STANDARD_ITER × two cameras × three dyes

Files are prefixed `ximea_` or `zwo_` to avoid collisions.  
`overwrite=False` allows resuming interrupted runs.

In [ ]:
cameras = [
    (validated_ximea, config_ximea, "ximea_"),
    (validated_zwo,   config_zwo,   "zwo_"),
]

strategy = FittingStrategy.STANDARD_ITER

for cam_params, cam_config, flag in cameras:
    for dye in dyes:
        print(f"Running {flag.strip('_')} / {dye} ...")
        try:
            MSF.test_simulation_method(
                dye=dye,
                filters=filters,
                wavelength=wavelength,
                camera_parameters=cam_params.__dict__.copy(),
                save_folder=save_folder,
                n_photon_space=n_photon_space,
                smoothing_function=smoothing_function,
                strategy=strategy,
                starting_flag=flag,
                config=cam_config,
                overwrite=False,
            )
            print(f"  Done.")
        except Exception as e:
            print(f"  FAILED: {e}")

print("All simulations complete.")

Running ximea / ATTO 488 ...
Analysed photon flux 200/200    Time elapsed: 276.716 min                       ?, ?it/s]
Completed analysis of 200 photon flux values    Total time: 276.716 min


INFO:src.Multicolour_Simulation_Functions:Simulation completed for strategy standard_iter


  Done.
Running ximea / ATTO 565 ...
Analysed photon flux 200/200    Time elapsed: 280.846 min                       ?, ?it/s]
Completed analysis of 200 photon flux values    Total time: 280.846 min


INFO:src.Multicolour_Simulation_Functions:Simulation completed for strategy standard_iter


  Done.
Running ximea / ATTO 647N ...
Analysed photon flux 200/200    Time elapsed: 285.871 min                       ?, ?it/s]
Completed analysis of 200 photon flux values    Total time: 285.871 min


INFO:src.Multicolour_Simulation_Functions:Simulation completed for strategy standard_iter


  Done.
Running zwo / ATTO 488 ...
Analysed photon flux 200/200    Time elapsed: 271.629 min                       ?, ?it/s]
Completed analysis of 200 photon flux values    Total time: 271.629 min


INFO:src.Multicolour_Simulation_Functions:Simulation completed for strategy standard_iter


  Done.
Running zwo / ATTO 565 ...


## Load results and compute precision metrics

Because `subtractx0y0=True`, the saved `xc` / `yc` columns are already *(fitted − truth)*, so:

$$\sigma_{xy} = \sqrt{\frac{\sigma_x^2 + \sigma_y^2}{2}} \times p_{\mathrm{px}}$$

where $p_{\mathrm{px}}$ is the per-camera pixel size in nm.  
Colour precision is the std dev of the Euclidean distance from each fitted $(A_B, A_G, A_R)$ point to the true colour vector.

In [ ]:
camera_labels = ["Ximea", "ZWO"]
camera_flags  = ["ximea_", "zwo_"]
camera_px_nm  = {"Ximea": 69.0, "ZWO": 71.5}   # pixel sizes in nm

parameters = ["xy", "colour", "B", "G", "R", "chi_mean", "chi_median", "chi_std"]

all_files = os.listdir(save_folder)

Overall_XR = xr.DataArray(
    data=np.full([len(camera_labels), len(dyes), len(n_photon_space), len(parameters)], np.nan),
    coords=[camera_labels, dyes, n_photon_space, parameters],
    dims=["Camera", "Dye", "Photon", "Metric"],
)

for i, (label, flag) in enumerate(zip(camera_labels, camera_flags)):
    px_nm = camera_px_nm[label]
    prefix = flag + "LM_method"
    flag_files = [x for x in all_files if x.startswith(prefix)]

    for j, dye in enumerate(dyes):
        dyestr = dye.replace("/", "-")
        raw_files = [
            os.path.join(save_folder, x)
            for x in flag_files
            if dyestr in x and "rawresults" in x
        ]
        param_files = [
            os.path.join(save_folder, x)
            for x in flag_files
            if dyestr in x and "input_parameters" in x
        ]
        if not raw_files or not param_files:
            print(f"Missing files for {label} / {dye} — skipping.")
            continue

        true_bgr = pd.read_csv(param_files[0]).to_numpy()[0][-3:]
        true_bgr = true_bgr / np.sum(true_bgr)
        true_colour = np.expand_dims(true_bgr, 0)

        results_all = pd.read_hdf(raw_files[0])

        for k, photonval in enumerate(n_photon_space):
            results = results_all[results_all["photon_level"] == k]
            if len(results) == 0:
                continue

            filt = ~(
                (results["s_x"] < 0)
                | (results["s_y"] < 0)
                | (results["xc"].abs() > 14)
                | (results["yc"].abs() > 14)
                | (results["chi_sqr"] > 6)
            )

            xc      = results["xc"].to_numpy()[filt]
            yc      = results["yc"].to_numpy()[filt]
            chi_sqr = results["chi_sqr"].to_numpy()[filt]

            sigma_xy = np.sqrt((np.nanstd(xc)**2 + np.nanstd(yc)**2) / 2) * px_nm

            colour = np.vstack([
                results["A_B"].to_numpy()[filt],
                results["A_G"].to_numpy()[filt],
                results["A_R"].to_numpy()[filt],
            ]).T
            colour_dist = cdist(colour, true_colour).ravel()
            B_dist = np.sqrt(np.square(colour[:, 0] - true_colour[:, 0]))
            G_dist = np.sqrt(np.square(colour[:, 1] - true_colour[:, 1]))
            R_dist = np.sqrt(np.square(colour[:, 2] - true_colour[:, 2]))

            Overall_XR[i, j, k, 0] = sigma_xy
            Overall_XR[i, j, k, 1] = np.nanstd(colour_dist)
            Overall_XR[i, j, k, 2] = np.nanstd(B_dist)
            Overall_XR[i, j, k, 3] = np.nanstd(G_dist)
            Overall_XR[i, j, k, 4] = np.nanstd(R_dist)
            Overall_XR[i, j, k, 5] = np.nanmean(chi_sqr)
            Overall_XR[i, j, k, 6] = np.nanmedian(chi_sqr)
            Overall_XR[i, j, k, 7] = np.nanstd(chi_sqr)

print("Results loaded.")
nc_path = os.path.join(save_folder, "ZWO_vs_Ximea_STDEV.nc")
Overall_XR.to_netcdf(nc_path)
print(f"Saved to {nc_path}")

In [ ]:
# (Optional) reload without re-running the simulation
Overall_XR = xr.load_dataarray(os.path.join(save_folder, "ZWO_vs_Ximea_STDEV.nc"))

In [ ]:
# Quick diagnostic — chi-squared distribution for the last loaded results slice
plt.hist(results["chi_sqr"], 100)
plt.yscale("log")
plt.xlabel(r"$\chi^2$")
plt.title("Chi-squared distribution (last dye/camera)")
plt.show()

## Plot — dye-averaged precision vs photon count

In [ ]:
plot_styles = [
    {"color": "#d40000", "ls": "-",  "lw": 1.2},   # Ximea
    {"color": "black",   "ls": "-.", "lw": 1.0},   # ZWO
]

fig_savefolder = "/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Papers/Multicolour/SI/"
os.makedirs(fig_savefolder, exist_ok=True)

fig, axs = plotter.two_column_plot(ncolumns=2, widthratio=np.ones(2))

for i, (label, style) in enumerate(zip(camera_labels, plot_styles)):
    mean_xy     = np.nanmean(Overall_XR[i, :, :, 0], axis=0)
    mean_colour = np.nanmean(Overall_XR[i, :, :, 1], axis=0)

    axs[0] = plotter.line_plot(
        axs[0], n_photon_space, mean_xy,
        label=label,
        xaxislabel=r"$N_{\mathrm{photons}}$",
        yaxislabel=r"$\sigma_{xy}$ / nm",
        **style,
    )
    axs[1] = plotter.line_plot(
        axs[1], n_photon_space, mean_colour,
        label=label,
        xaxislabel=r"$N_{\mathrm{photons}}$",
        yaxislabel=r"$\sigma_{\mathrm{colour}}$ / RGB",
        **style,
    )

for ax in axs:
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim([500, 50000])

axs[0].set_ylim([0.3, 30])
axs[0].set_xlabel("")
axs[1].set_ylim([5e-3, 0.1])
axs[0].legend(loc="best", fontsize=6)

plt.savefig(os.path.join(fig_savefolder, "ZWO_vs_Ximea.svg"), dpi=600, format="svg")
plt.show()

In [ ]:
# Per-channel (B, G, R) colour precision
fig, axs = plotter.two_column_plot(ncolumns=3, widthratio=np.ones(3))

for i, (label, style) in enumerate(zip(camera_labels, plot_styles)):
    mean_B = np.nanmean(Overall_XR[i, :, :, 2], axis=0)
    mean_G = np.nanmean(Overall_XR[i, :, :, 3], axis=0)
    mean_R = np.nanmean(Overall_XR[i, :, :, 4], axis=0)

    axs[0] = plotter.line_plot(
        axs[0], n_photon_space, mean_B, label=label,
        xaxislabel=r"$N_{\mathrm{photons}}$", yaxislabel=r"$\sigma_B$ / RGB", **style,
    )
    axs[1] = plotter.line_plot(
        axs[1], n_photon_space, mean_G, label=label,
        xaxislabel=r"$N_{\mathrm{photons}}$", yaxislabel=r"$\sigma_G$ / RGB", **style,
    )
    axs[2] = plotter.line_plot(
        axs[2], n_photon_space, mean_R, label=label,
        xaxislabel=r"$N_{\mathrm{photons}}$", yaxislabel=r"$\sigma_R$ / RGB", **style,
    )

for ax in axs:
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim([500, 50000])

axs[0].legend(loc="best", fontsize=6)

plt.savefig(os.path.join(fig_savefolder, "ZWO_vs_Ximea_RGB.svg"), dpi=600, format="svg")
plt.show()

In [ ]:
# Chi-squared goodness-of-fit
fig, axs = plotter.two_column_plot(ncolumns=3, widthratio=np.ones(3))

for i, (label, style) in enumerate(zip(camera_labels, plot_styles)):
    axs[0] = plotter.line_plot(
        axs[0], n_photon_space, np.nanmean(Overall_XR[i, :, :, 5], axis=0),
        label=label, xaxislabel=r"$N_{\mathrm{photons}}$", yaxislabel=r"$\chi^2$ mean", **style,
    )
    axs[1] = plotter.line_plot(
        axs[1], n_photon_space, np.nanmean(Overall_XR[i, :, :, 6], axis=0),
        label=label, xaxislabel=r"$N_{\mathrm{photons}}$", yaxislabel=r"$\chi^2$ median", **style,
    )
    axs[2] = plotter.line_plot(
        axs[2], n_photon_space, np.nanmean(Overall_XR[i, :, :, 7], axis=0),
        label=label, xaxislabel=r"$N_{\mathrm{photons}}$", yaxislabel=r"$\chi^2$ std", **style,
    )

for ax in axs:
    ax.set_xscale("log")
    ax.set_xlim([500, 50000])

axs[0].legend(loc="best", fontsize=6)

plt.savefig(os.path.join(fig_savefolder, "ZWO_vs_Ximea_Chi.svg"), dpi=600, format="svg")
plt.show()

## Per-dye breakdown

Show each dye separately to check whether the camera ranking is consistent across the spectrum.

In [ ]:
fig, axs = plt.subplots(
    2, len(dyes),
    figsize=(3.2535433 * len(dyes) / 2, 3.1909449),
    sharex=True,
)

for j, dye in enumerate(dyes):
    for i, (label, style) in enumerate(zip(camera_labels, plot_styles)):
        axs[0, j].plot(
            n_photon_space, Overall_XR[i, j, :, 0],
            label=label if j == 0 else None,
            **style,
        )
        axs[1, j].plot(
            n_photon_space, Overall_XR[i, j, :, 1],
            **style,
        )
    axs[0, j].set_title(dye, fontsize=7)

for row in range(2):
    for col in range(len(dyes)):
        axs[row, col].set_xscale("log")
        axs[row, col].set_yscale("log")
        axs[row, col].set_xlim([500, 50000])

for col in range(len(dyes)):
    axs[0, col].set_ylim([0.3, 30])
    axs[1, col].set_ylim([5e-3, 0.3])
    axs[1, col].set_xlabel(r"$N_{\mathrm{photons}}$", fontsize=7)

axs[0, 0].set_ylabel(r"$\sigma_{xy}$ / nm", fontsize=7)
axs[1, 0].set_ylabel(r"$\sigma_{\mathrm{colour}}$ / RGB", fontsize=7)
axs[0, 0].legend(loc="best", fontsize=5)

plt.tight_layout()
plt.savefig(os.path.join(fig_savefolder, "ZWO_vs_Ximea_PerDye.svg"), dpi=600, format="svg")
plt.show()

## Ratio plot — ZWO relative to Ximea

Values > 1 mean ZWO is worse; values < 1 mean ZWO is better.  
Ximea (red) is the reference line at 1.

In [ ]:
fig, axs = plotter.one_column_plot(npanels=2, height=3.1909449, ratios=[1, 1], width=3.2535433)

ref_xy     = np.nanmean(Overall_XR[0, :, :, 0], axis=0)  # Ximea reference
ref_colour = np.nanmean(Overall_XR[0, :, :, 1], axis=0)

zwo_xy     = np.nanmean(Overall_XR[1, :, :, 0], axis=0)
zwo_colour = np.nanmean(Overall_XR[1, :, :, 1], axis=0)

axs[0] = plotter.line_plot(
    axs[0], n_photon_space, zwo_xy / ref_xy,
    label="ZWO",
    xaxislabel=r"$N_{\mathrm{photons}}$",
    yaxislabel=r"$\sigma_{xy}^{\mathrm{ZWO}}$ / $\sigma_{xy}^{\mathrm{Ximea}}$",
    color="black", ls="-.", lw=1.0,
)
axs[1] = plotter.line_plot(
    axs[1], n_photon_space, zwo_colour / ref_colour,
    label="ZWO",
    xaxislabel=r"$N_{\mathrm{photons}}$",
    yaxislabel=r"$\sigma_{\mathrm{colour}}^{\mathrm{ZWO}}$ / $\sigma_{\mathrm{colour}}^{\mathrm{Ximea}}$",
    color="black", ls="-.", lw=1.0,
)

for ax in axs:
    ax.set_xscale("log")
    ax.set_xlim([500, 50000])
    ax.axhline(1.0, color="#d40000", lw=0.8, ls="-", label="Ximea (reference)")

axs[0].set_xlabel("")
axs[0].legend(loc="best", fontsize=6)

plt.savefig(os.path.join(fig_savefolder, "ZWO_vs_Ximea_Ratio.svg"), dpi=600, format="svg")
plt.show()